[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/p3_generation/09_capital_and_lcoe.ipynb)

# What does it cost to build a power plant?
## REE 4301 / IE 5300 - Energy Systems Modeling

Work through this in order. Each step is one line of arithmetic, and each one sets up the next question.

Two plants, both **250 MW**, both assumed to last **30 years**: one coal, one gas.


## Setup

One cell, the same in every notebook here: it installs what Colab does not have, fetches this repository so `data/` and `src/` are present, and moves into this notebook's own folder so the relative paths below resolve. The notebook's own imports follow in the same cell.


In [1]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/p3_generation"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = []

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---

import pandas as pd

working directory: p3_generation


---
## 1. How much electricity?

If the unit ran flat out for its whole life, how many MWh would it produce? Work it out before you run the cell.


In [2]:
mw, hours, years = 250, 8760, 30
lifetime_mwh = mw * hours * years

print(f'{lifetime_mwh:,} MWh   =  {lifetime_mwh / 1e6:.1f} TWh')


65,700,000 MWh   =  65.7 TWh


## 2. What is that worth?

Suppose you sell every MWh at a flat price. Compute the lifetime revenue at **$10/MWh** and at **$50/MWh**.

Both are prices you can find in a real market, in different hours.


In [3]:
for price in (10, 50):
    print(f'at ${price:>2}/MWh   ${lifetime_mwh * price / 1e6:>9,.0f} M')


at $10/MWh   $      657 M
at $50/MWh   $    3,285 M


---
## 3. How much fuel does that take?

Heat rate is how much fuel energy you burn per unit of electricity out, in Btu/kWh. A **lower** number is a more efficient plant.

Pick three that bracket what real units achieve - a good one, a middling one, and a tired one. The ones below are reasonable; substitute your own and cite them.

Then convert fuel energy into the unit the fuel is actually sold in: **tons** for coal, **thousand cubic feet** for gas. That conversion is the heat content.


In [4]:
# MMBtu of fuel per MWh of electricity  =  heat rate / 1000
coal_heat_rates = [8_800, 10_000, 11_500]     # Btu/kWh
gas_heat_rates = [6_400, 7_500, 9_000]

COAL_HEAT_CONTENT = 13.0      # MMBtu per ton, Texas lignite
GAS_HEAT_CONTENT = 1.036      # MMBtu per Mcf

rows = []
for hr in coal_heat_rates:
    mmbtu = lifetime_mwh * hr / 1000
    rows.append({'fuel': 'coal', 'heat rate': hr,
                 'MMBtu': round(mmbtu / 1e6, 1),
                 'fuel units': f'{mmbtu / COAL_HEAT_CONTENT / 1e6:,.1f} Mt'})
for hr in gas_heat_rates:
    mmbtu = lifetime_mwh * hr / 1000
    rows.append({'fuel': 'gas', 'heat rate': hr,
                 'MMBtu': round(mmbtu / 1e6, 1),
                 'fuel units': f'{mmbtu / GAS_HEAT_CONTENT / 1e6:,.0f} Bcf'})

print(pd.DataFrame(rows).to_string(index=False))
print('\nMMBtu column is in millions.')


fuel  heat rate  MMBtu fuel units
coal       8800  578.2    44.5 Mt
coal      10000  657.0    50.5 Mt
coal      11500  755.5    58.1 Mt
 gas       6400  420.5    406 Bcf
 gas       7500  492.8    476 Bcf
 gas       9000  591.3    571 Bcf

MMBtu column is in millions.


> Look at the coal tonnage for a moment. How many rail cars is that, and how often would one have to arrive?


---
## 4. What does the fuel cost?

Three prices each, low to high. Fuel prices move a great deal, so a single number would be a guess dressed up as an answer.


In [5]:
coal_prices = [15, 20, 30]          # $/ton
gas_prices = [2.00, 2.50, 4.00]     # $/Mcf

def fuel_bill(hr, heat_content, price):
    units = (lifetime_mwh * hr / 1000) / heat_content
    return units * price

coal = pd.DataFrame(
    [[fuel_bill(hr, COAL_HEAT_CONTENT, p) / 1e6 for p in coal_prices]
     for hr in coal_heat_rates],
    index=[f'{hr} Btu/kWh' for hr in coal_heat_rates],
    columns=[f'${p}/ton' for p in coal_prices]).round(0)

gas = pd.DataFrame(
    [[fuel_bill(hr, GAS_HEAT_CONTENT, p) / 1e6 for p in gas_prices]
     for hr in gas_heat_rates],
    index=[f'{hr} Btu/kWh' for hr in gas_heat_rates],
    columns=[f'${p:.2f}/Mcf' for p in gas_prices]).round(0)

print('lifetime fuel bill, $ millions\n')
print('COAL'); print(coal.to_string()); print()
print('GAS');  print(gas.to_string())


lifetime fuel bill, $ millions

COAL
               $15/ton  $20/ton  $30/ton
8800 Btu/kWh     667.0    889.0   1334.0
10000 Btu/kWh    758.0   1011.0   1516.0
11500 Btu/kWh    872.0   1162.0   1744.0

GAS
              $2.00/Mcf  $2.50/Mcf  $4.00/Mcf
6400 Btu/kWh      812.0     1015.0     1623.0
7500 Btu/kWh      951.0     1189.0     1903.0
9000 Btu/kWh     1142.0     1427.0     2283.0


## 5. Compare

Put the two tables next to your revenue figures from step 2.

At **$50/MWh**, how many of those fuel bills can you cover?

At **$10/MWh**, how many?


---
## 6. Where do you break even on fuel alone?

Divide the lifetime fuel bill by the lifetime MWh. That gives the price below which the plant would rather not run at all - roughly the number it would bid into an energy-only market.


In [6]:
rows = []
for hr, p in zip(coal_heat_rates, coal_prices):
    rows.append({'plant': f'coal {hr} Btu/kWh @ ${p}/ton',
                 '$/MWh': round(fuel_bill(hr, COAL_HEAT_CONTENT, p)
                                / lifetime_mwh, 2)})
for hr, p in zip(gas_heat_rates, gas_prices):
    rows.append({'plant': f'gas {hr} Btu/kWh @ ${p:.2f}/Mcf',
                 '$/MWh': round(fuel_bill(hr, GAS_HEAT_CONTENT, p)
                                / lifetime_mwh, 2)})

print(pd.DataFrame(rows).to_string(index=False))


                       plant  $/MWh
 coal 8800 Btu/kWh @ $15/ton  10.15
coal 10000 Btu/kWh @ $20/ton  15.38
coal 11500 Btu/kWh @ $30/ton  26.54
gas 6400 Btu/kWh @ $2.00/Mcf  12.36
gas 7500 Btu/kWh @ $2.50/Mcf  18.10
gas 9000 Btu/kWh @ $4.00/Mcf  34.75


> The spread between the cheapest and dearest row is roughly a factor of three, and you produced all of it by choosing assumptions. Which of the two - heat rate or fuel price - moved it more?

> And you have not paid for the plant yet.


---
## 7. Now add the plant itself

Look up overnight capital cost and fixed O&M for each technology in the **NREL Annual Technology Baseline**. Cite the year you used.

Spread the capital over the lifetime with a capital recovery factor - a mortgage payment on the plant:

&nbsp;&nbsp;&nbsp;&nbsp;CRF = *r*(1+*r*)<sup>*n*</sup> / ((1+*r*)<sup>*n*</sup> − 1)

Then put the three pieces over the same denominator:

&nbsp;&nbsp;&nbsp;&nbsp;(capital + fixed O&M + fuel) / total MWh&nbsp;&nbsp; = &nbsp;$/MWh

**Tell me what number I have to sell electricity at to get my money back on each plant.**


In [7]:
# Fill these in from the ATB you cite. The values below are placeholders
# so the cell runs - replace them and say which edition they came from.
overnight = {'coal': 3_800_000.0, 'gas': 1_050_000.0}    # $/MW
fom = {'coal': 45_000.0, 'gas': 15_300.0}                # $/MW-yr
rate, life = 0.07, 30

crf = rate * (1 + rate) ** life / ((1 + rate) ** life - 1)
print(f'CRF = {crf:.4f}\n')

chosen = {'coal': (10_000, COAL_HEAT_CONTENT, 20),
          'gas': (7_500, GAS_HEAT_CONTENT, 2.50)}

rows = []
for tech, (hr, hc, price) in chosen.items():
    cap = overnight[tech] * crf * mw * years
    fixed = fom[tech] * mw * years
    fuel = fuel_bill(hr, hc, price)
    rows.append({'plant': tech,
                 'capital $M': round(cap / 1e6),
                 'FOM $M': round(fixed / 1e6),
                 'fuel $M': round(fuel / 1e6),
                 'break-even $/MWh':
                     round((cap + fixed + fuel) / lifetime_mwh, 2)})

print(pd.DataFrame(rows).to_string(index=False))


CRF = 0.0806

plant  capital $M  FOM $M  fuel $M  break-even $/MWh
 coal        2297     338     1011             55.48
  gas         635     115     1189             29.50


> Compare those two numbers against the fuel-only figures from step 6, and against the $10 and $50 prices you started with.

> One of these plants has most of its cost before it burns anything, and the other has most of its cost afterwards. Which is which, and what would that mean for how often each one wants to run?


---
### Before class

*You priced this as a plant that sells into a market. If the same 250 MW sat behind one customer's meter and never sold a thing, which number in your tables would change, and by how much?*


---
### Does the package agree?

`esm.lcoe` recomputes everything above from `data/raw/`. There is no solver here, so agreement cannot mean two searches found the same optimum - but there *is* a genuinely different second route, and it is the one worth your attention.

**What you did above** was annualise the capital with a CRF, add fuel and fixed costs, and divide the nominal total by the nominal lifetime energy. **The definition of LCOE** is not that. It is discounted cost over discounted energy:

$$	ext{LCOE} = rac{\sum_t C_t/(1+r)^t}{\sum_t E_t/(1+r)^t}$$

Those are different calculations. They agree here **only because the annual energy is constant over the life** - both sums pick up the same annuity factor and it cancels. That cancellation is the whole justification for the shortcut every screening study uses, and it is almost always assumed rather than shown.

> **Predict before you run it.** A solar array that loses half a percent of its output every year breaks that cancellation. Before you look: does the true LCOE come out above or below what the shortcut reports, and roughly by how much over thirty years?


In [8]:
from esm.lcoe import (capital_recovery_factor, lcoe_by_crf, lcoe_by_dcf,
                     load_lcoe_instance, screening_table)
from esm.tolerance import AGREEMENT_RTOL, relative

inst = load_lcoe_instance()
pkg_screen = screening_table(inst)

checks = [('CRF', crf, capital_recovery_factor(rate, life))]
checks += [(f'{f} {hr} $/MWh',
            fuel_bill(hr, COAL_HEAT_CONTENT if f == 'coal'
                      else GAS_HEAT_CONTENT, p) / lifetime_mwh,
            pkg_screen[(f, hr)])
           for f, hr, p in
           [('coal', h, q) for h, q in zip(coal_heat_rates, coal_prices)]
           + [('gas', h, q) for h, q in zip(gas_heat_rates, gas_prices)]]
# the notebook's OWN computed values, not the 2-dp figures it printed:
# a displayed number is rounded, and AGREEMENT_RTOL is 1e-9.
nb_breakeven = {t: ((overnight[t] * crf * mw * years
                    + fom[t] * mw * years
                    + fuel_bill(*chosen[t])) / lifetime_mwh)
                for t in chosen}
checks += [(f'{t} break-even', nb_breakeven[t],
            lcoe_by_crf(inst, inst.plants[(t, chosen[t][0])]))
           for t in chosen]

print(f'{"quantity":24s} {"notebook":>12s} {"package":>12s} {"rel diff":>10s}')
for label, hand, pkg in checks:
    print(f'{label:24s} {hand:12.4f} {pkg:12.4f} {relative(hand, pkg):10.1e}')

worst = max(relative(a, b) for _, a, b in checks)
assert worst < AGREEMENT_RTOL, (
    f'notebook and package disagree by {worst:.2e}, '
    f'which is worse than {AGREEMENT_RTOL:.0e}')
print()
print(f'notebook and package agree to {worst:.1e}')


quantity                     notebook      package   rel diff
CRF                            0.0806       0.0806    0.0e+00
coal 8800 $/MWh               10.1538      10.1538    0.0e+00
coal 10000 $/MWh              15.3846      15.3846    0.0e+00
coal 11500 $/MWh              26.5385      26.5385    0.0e+00
gas 6400 $/MWh                12.3552      12.3552    0.0e+00
gas 7500 $/MWh                18.0985      18.0985    0.0e+00
gas 9000 $/MWh                34.7490      34.7490    0.0e+00
coal break-even               55.4792      55.4792    0.0e+00
gas break-even                29.5044      29.5044    0.0e+00

notebook and package agree to 0.0e+00


Now the part that is not a restatement.


In [9]:
gas = inst.plants[('gas', 7500)]

shortcut = lcoe_by_crf(inst, gas)
definition = lcoe_by_dcf(inst, gas)
degrading = lcoe_by_dcf(inst, gas, degradation=0.005)

print(f'CRF shortcut, flat output      ${shortcut:6.2f}/MWh')
print(f'DCF definition, flat output    ${definition:6.2f}/MWh')
print(f'DCF definition, 0.5%/yr loss   ${degrading:6.2f}/MWh')
print()
print(f'the two routes agree to {relative(shortcut, definition):.1e}'
      f' on flat output,')
print(f'and diverge by {degrading / definition - 1:.1%} once it degrades.')
print()
print('The shortcut cannot see the degradation: it divides a nominal')
print('total by a nominal total, and both fell by the same factor.')
print('That is the limit of the method you just used - and the reason')
print("a real study discounts the ENERGY as well as the money.")


CRF shortcut, flat output      $ 29.50/MWh
DCF definition, flat output    $ 29.50/MWh
DCF definition, 0.5%/yr loss   $ 30.96/MWh

the two routes agree to 0.0e+00 on flat output,
and diverge by 4.9% once it degrades.

The shortcut cannot see the degradation: it divides a nominal
total by a nominal total, and both fell by the same factor.
That is the limit of the method you just used - and the reason
a real study discounts the ENERGY as well as the money.


### Bonus

Use what you just computed to parameterise the three-node PyPSA example: replace its gas figures with your real ones, and add a coal plant with yours. Then see whether the solver builds the thing you would have.

### Sources
- NREL Annual Technology Baseline - overnight cost, fixed O&M, lifetimes. Cite the edition.
- EIA for fuel prices and heat contents; both vary by region and year.
